# Execute and inspect Couchbase queries

**Goal:** compare SQL++ execution, plans and profiling, then run a small workload.

Initialize the [Couchbase Community demo](../docs/deployment.md#demo-data) and install
`uv sync --extra couchbase`. Run cells in order with the BONDI Python environment.
The local Docker connection and credentials are defaults; optional `COUCHBASE_URI`,
`COUCHBASE_USERNAME` and `COUCHBASE_PASSWORD` overrides select another deployment.
All examples read the eight fictional demo businesses without modifying them.

Companion guides: [execution](../docs/executor.md), [summaries](../docs/summaries.md).

In [ ]:
import os
from dataclasses import asdict
from pprint import pprint

from couchbase.n1ql import QueryScanConsistency

from bondi.couchbase import (
    CouchbaseExecutor,
    CouchbaseQuery,
    summarize,
)
from bondi.workload import Workload, WorkloadEntry

# For the optional Enterprise environment, set COUCHBASE_URI to port 11211.
uri = os.environ.get("COUCHBASE_URI", "couchbase://127.0.0.1:11210?network=external")
options = {"read_only": True, "scan_consistency": QueryScanConsistency.REQUEST_PLUS}


def connect():
    """Create a local executor; use it in a with block to ensure cleanup."""
    return CouchbaseExecutor.from_uri(
        uri,
        username=os.environ.get("COUCHBASE_USERNAME", "Administrator"),
        password=os.environ.get("COUCHBASE_PASSWORD", "bondi-local"),
        concurrency=2,
        continue_on_error=True,
        timeout_seconds=10,
    )


city_query = CouchbaseQuery(
    "SELECT RAW b.name FROM business b WHERE city = $city ORDER BY b.name",
    named_parameters={"city": "Sydney"},
    query_context="demo._default",
)
count_query = CouchbaseQuery("SELECT RAW COUNT(*) FROM `demo`.`_default`.`business`")
workload = Workload(
    [WorkloadEntry(city_query, pattern_id=0), WorkloadEntry(count_query, pattern_id=1)]
)

## One query, three operations

`run` consumes rows and optionally retains them. `explain` returns a plan without
executing the original query. `profile` executes with diagnostic timing enabled;
it can retain rows too. Community returns basic metrics and a warning about missing
detailed profiling; Enterprise adds an operator tree. No extra EXPLAIN query is issued.


In [ ]:
with connect() as executor:
    result = executor.run(city_query, collect_results=True, query_kwargs=options)
    plan = executor.explain(city_query, query_kwargs=options)
    measured = executor.profile(city_query, collect_results=True, query_kwargs=options)

print("Names:", result.documents)
print("Consumed rows:", result.row_count)
print("Same profile rows:", measured.documents == result.documents)
pprint(plan.raw["plan"], depth=3)
pprint(measured.raw["metrics"])
print("Warnings:", measured.warnings)

## Workload outcomes

All three workload methods return input-aligned lists. `None` means an entry was
never started. Expected database failures appear in `.error`; programming errors
still raise. With concurrent execution, already-running entries finish after an error.


In [ ]:
with connect() as executor:
    runs = executor.run_workload(workload, query_kwargs=options)
    plans = executor.explain_workload(workload, query_kwargs=options)
    profiles = executor.profile_workload(workload, query_kwargs=options)

for name, results in (("run", runs), ("explain", plans), ("profile", profiles)):
    print(name, [(r.status, r.row_count) if r is not None else None for r in results])

assert all(r is not None and r.error is None for group in (runs, plans, profiles) for r in group)

# EXPLAIN rows describe plans, not query output, so they have no row_count.
assert plans[0].row_count is None
assert runs[0].row_count == profiles[0].row_count == 3
# SELECT RAW COUNT(*) returns one scalar row; it does not return eight rows.
assert runs[1].row_count == 1

## Inspect the native diagnostics

Metadata uses plain JSON values and milliseconds. The profile tree keeps Couchbase's
native operator names and counters. An emitted index item is not automatically a
MongoDB-style examined index entry. The summary leaves examination totals unavailable.

`REQUEST_PLUS` waits for fresh index contents; that wait affects measurements.
Try `NOT_BOUNDED` only when you accept potentially stale index contents.


In [ ]:
profile_tree = (measured.raw["profile"] or {}).get("executionTimings")
if profile_tree is None:
    print("No detailed operator profile; basic metrics are still available.")
else:
    pprint(profile_tree, depth=3)

## Parse common measurements explicitly
`summarize()` detects the raw format and works offline without the SDK. EXPLAIN supplies plan
information; run/profile metadata supplies server time and output counts. Detailed
profiles also supply a native operator sequence, which can differ from EXPLAIN.


In [ ]:
plan_summary = summarize(plan.raw)
execution_summary = summarize(measured.raw)
pprint(asdict(plan_summary))
pprint(asdict(execution_summary))
assert plan_summary.rows_returned is None
assert execution_summary.rows_returned == result.row_count == 3
assert execution_summary.keys_examined is None

# Workload results stay aligned; skip failures and unstarted entries.
summary_rows = [
    asdict(summarize(r.raw)) if r is not None and r.status == "success" else None for r in profiles
]
pprint(summary_rows)

## Try next

Change the city parameter and compare returned rows and plans. Try
`SELECT RAW b` versus `SELECT b` to see why row shape matters for schema inspection.
A `COUNT(*)` result is one output row even when the count inside it is larger.

Missing detailed profile information on Community is expected, not a failed query.
The [API reference](../docs/reference/couchbase/executors.md) covers native options
and advanced asynchronous integration.